# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzlanFaisalRaj/flyrank-internship-ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook is the working version of the deployed research paper (`docs/index.html`). Each
section below matches a section of the paper. Numbers here are recomputed from
`data/raw/content_refresh_anonymized.csv` and match the weekly notebooks
(`w03`–`w07`) they build on.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load
> `writing-research-papers/SKILL.md` and `deploying-static-pages/SKILL.md`.

## 1. Question

**Research question.** Among a client's content pages, which ones are worth reviewing for a
refresh first, and why?

**The decision this supports.** A content strategist or SEO reviewer running a weekly or
biweekly refresh triage picks a shortlist to look at out of a much larger library they don't
have time to review page by page.

**Cost of a wrong call.** A false positive wastes an editor's time refreshing a page that
didn't need it. A false negative lets a real decline run unreviewed until the next audit cycle.
Neither is catastrophic on its own, which is why this is framed as decision-support — a ranked
shortlist a human still checks — not an automated action.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Rows: {len(df):,}  |  Clients: {df['client_id'].nunique()}  |  Columns: {df.shape[1]}")
print(f"Base rate (declining): {df['is_declining_label'].mean():.3f}")

Rows: 30,000  |  Clients: 32  |  Columns: 45
Base rate (declining): 0.542


## 2. Data

**Source and release.** `data/raw/content_refresh_anonymized.csv` — the FlyRank ML Internship
starter dataset, 30,000 rows, one row per content item, aggregated over a trailing 90-day
window plus a last-30-days vs prior-30-days comparison. 32 pseudonymous clients, hashed IDs
only. This same student also queried the full production warehouse
(`hf://datasets/FlyRank/internship-warehouse`, the March 2026 `fact_content_daily_performance`
partition) directly in `work/notebooks/w03_data_contract.ipynb` to build and verify a daily-grain
data contract; the capstone model below trains on the 30,000-row aggregated release because it
is the one with a usable label and full weekly-notebook lineage (ML-07 through ML-10).

**What's excluded and why.**
- `trend_direction` / `trend_pct` — these define the label; used as the label, never as a feature.
- Every `*_last_30d` / `*_prev_30d` column — the label is literally the swing between these two
  windows, so any of them in the feature set would leak the answer (confirmed directly in
  Section 3 below).
- `content_id` / `client_id` — grouping keys only, never features.
- `gsc_avg_position == 0` rows, when working from the warehouse — `0` means "no position data,"
  not rank zero.

**Public safety.** No client names, raw queries, or identifying details appear anywhere in this
notebook or the deployed paper — only hashed IDs, and those are used solely for grouping.

In [2]:
window_cols = [c for c in df.columns if c.endswith("_last_30d") or c.endswith("_prev_30d")]
drop_cols = {"content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"} | set(window_cols)
feature_cols = [c for c in df.columns if c not in drop_cols]
num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c])]

print(f"Excluded window columns ({len(window_cols)}): {window_cols}")
print(f"Feature columns kept: {len(feature_cols)}  ({len(num_cols)} numeric, {len(cat_cols)} categorical)")

Excluded window columns (6): ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
Feature columns kept: 34  (23 numeric, 11 categorical)


## 3. Methodology

**Label.** `is_declining_label` = 1 if `trend_direction == "down"` (a >10% impression drop,
last 30 days vs prior 30 days), else 0. This is an **observed proxy**, not a guaranteed future
outcome.

**Baseline (Week 4 / ML-07).** A transparent rule: flag a page if it is stale
(`freshness_tier == "91-180"`, the one window a signal check found genuinely elevated — 61.1%
decline rate vs a 54.2% base rate), visible (`impressions_90d >= 300`), and underperforming its
own position tier's expected CTR (computed with a volume floor, so low-traffic noise doesn't
distort the tier medians).

**Model (Week 5 / ML-08).** Logistic Regression and Random Forest, compared on the same split
and metric as the baseline. Two models, not more — complexity has to earn its place on the
metric that matters (precision@K for a top-of-queue reviewer), not be added by default.

**Validation design (Week 6 / ML-09).** Split by `client_id` (`GroupShuffleSplit`, 75/25, seed
42), not row-by-row. Content items from the same client share client-level patterns, so a random
row split would leak client identity across train/test and inflate the score. There is no usable
timestamp column in this release to build a time-aware split on, so client-grouped is the
honest option available. Client overlap between train and test is confirmed at 0 below.

**Leakage checks.** The window-column exclusion above was tested directly: adding
`impressions_last_30d`/`impressions_prev_30d` back in pushes precision@10 from 0.80 to 1.00 and
ROC AUC from 0.575 to 0.809 — exactly the "collapse when removed" signature of a label-derived
leak. Their absence from the final feature set is doing real work, not just following a rule by
habit.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

print(f"train rows: {len(train):,} | test rows: {len(test):,}")
print(f"train clients: {train['client_id'].nunique()} | test clients: {test['client_id'].nunique()}")
print(f"client overlap (must be 0): {len(set(train['client_id']) & set(test['client_id']))}")
print(f"train base rate: {train['is_declining_label'].mean():.3f} | test base rate: {test['is_declining_label'].mean():.3f}")

cat_pipe = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))])
pre_rf = ColumnTransformer([("num", SimpleImputer(strategy="median"), num_cols),
                             ("cat", cat_pipe, cat_cols)])
pre_logreg = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
    ("cat", cat_pipe, cat_cols)])

logreg = Pipeline([("pre", pre_logreg), ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))])
rf = Pipeline([("pre", pre_rf), ("clf", RandomForestClassifier(n_estimators=300, max_depth=8,
                random_state=RANDOM_SEED, n_jobs=-1))])

logreg.fit(train[feature_cols], train["is_declining_label"])
rf.fit(train[feature_cols], train["is_declining_label"])
print("Models trained.")

train rows: 22,885 | test rows: 7,115
train clients: 24 | test clients: 8
client overlap (must be 0): 0
train base rate: 0.550 | test base rate: 0.517


Models trained.


## 4. Results (vs baseline)

Same test split, same metric, for the baseline rule and both models. Base rate (54.2%) is the
coin-flip floor any of these numbers have to beat to mean anything.

In [4]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    return float(np.asarray(y_true)[order][:k].mean())

# Baseline rule, recomputed on THIS test split (fit its thresholds on train only)
floor = train[train["impressions_90d"] >= 300]
expected_ctr_by_tier = floor.groupby("position_tier")["ctr"].median()
test_b = test.copy()
test_b["expected_ctr"] = test_b["position_tier"].map(expected_ctr_by_tier)
stale = test_b["freshness_tier"] == "91-180"
visible = test_b["impressions_90d"] >= 300
ctr_gap = test_b["ctr"] < test_b["expected_ctr"]
test_b["baseline_score"] = (stale.astype(int) * visible.astype(int) *
                             test_b["impressions_90d"] * (1 + ctr_gap.astype(int)))

results = {}
base_scores = test_b["baseline_score"].values
results["baseline_rule"] = {
    "p@10": precision_at_k(test["is_declining_label"].values, base_scores, 10),
    "p@20": precision_at_k(test["is_declining_label"].values, base_scores, 20),
    "roc_auc": roc_auc_score(test["is_declining_label"], base_scores),
}

for name, model in [("logistic_regression", logreg), ("random_forest", rf)]:
    proba = model.predict_proba(test[feature_cols])[:, 1]
    results[name] = {
        "p@10": precision_at_k(test["is_declining_label"].values, proba, 10),
        "p@20": precision_at_k(test["is_declining_label"].values, proba, 20),
        "roc_auc": roc_auc_score(test["is_declining_label"], proba),
    }

results_df = pd.DataFrame(results).T.round(3)
results_df.loc["base_rate (floor)"] = [round(test["is_declining_label"].mean(), 3), None, None]
print(results_df)

                      p@10  p@20  roc_auc
baseline_rule        0.300   0.3    0.489
logistic_regression  0.800   0.7    0.575
random_forest        0.500   0.5    0.597
base_rate (floor)    0.517   NaN      NaN


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

os.makedirs("work/figures", exist_ok=True)

plot_df = results_df.drop("base_rate (floor)")[["p@10", "p@20"]].astype(float)
fig, ax = plt.subplots(figsize=(6.5, 4))
plot_df.plot(kind="bar", ax=ax, color=["#2b6cb0", "#63b3ed"])
ax.axhline(test["is_declining_label"].mean(), color="#c53030", linestyle="--",
           label=f"base rate ({test['is_declining_label'].mean():.2f})")
ax.set_ylabel("precision")
ax.set_title("Model vs baseline vs base rate (client-grouped test split)")
ax.legend()
plt.xticks(rotation=20)
fig.tight_layout()
fig.savefig("work/figures/capstone_model_vs_baseline.png", dpi=130)
plt.close(fig)
print("Saved work/figures/capstone_model_vs_baseline.png")

Saved work/figures/capstone_model_vs_baseline.png


**Reading the table.** Both models beat the baseline rule and the base rate on precision@10.
Logistic Regression is the stronger top-of-queue ranker (higher precision@10 and precision@20);
Random Forest edges it out on overall ROC AUC but doesn't translate that into a better top-10.
Because this queue is read top-down by a reviewer with limited time, precision@K is the metric
that matters here, and the simpler model wins on it. That's the finding, not a footnote: more
complexity did not earn its place.

## 5. Limitations

- **One static snapshot, no time-aware split.** This release has no usable timestamp column, so
  validation is client-grouped, not time-aware. A page's future behavior a month from now is not
  directly tested.
- **Observed proxy, not a real outcome.** `is_declining_label` is a same-window bucket
  (`trend_direction`), not a verified future decline. It mirrors the lane's beginner proxy, not a
  causal or predictive ground truth.
- **Modest discrimination.** ROC AUC sits at 0.575–0.60 — a real lift over guessing, far from a
  reliable oracle. On the model's own numbers, roughly 2 in 10 top-10 picks are expected to be
  wrong.
- **Traffic-weighted queue.** Because `impressions_90d` drives both the baseline rule and the
  playbook's priority score, high-traffic clients structurally dominate the top of the queue. A
  client with less overall traffic but a genuinely urgent problem can rank lower than its urgency
  warrants.
- **No query-level or SERP context.** A low CTR here could mean a stale page, a branded query, a
  mid-run title experiment, or a SERP feature stealing the click — this data can't tell those
  apart; a human has to check.
- **Cross-sectional, not causal.** Nothing here says refreshing a page *will cause* it to
  recover — only that it looks worth reviewing first, and why.

## 6. Ranked recommendations

The action playbook (Week 7 / ML-10) ranks the honest-split model's test rows by
`priority_score = model_probability x impressions_90d`, so a page only ranks high if the model
flags real risk **and** enough people actually see it.

In [6]:
test_p = test.copy()
test_p["model_proba"] = logreg.predict_proba(test_p[feature_cols])[:, 1]
test_p["expected_ctr"] = test_p["position_tier"].map(expected_ctr_by_tier)
test_p["stale"] = test_p["freshness_tier"] == "91-180"
test_p["visible"] = test_p["impressions_90d"] >= 300
test_p["ctr_gap"] = test_p["ctr"] < test_p["expected_ctr"]
test_p["low_engagement"] = (test_p["engagement_rate"] <
                             test_p.groupby("position_tier")["engagement_rate"].transform("median"))

def reason_codes(row):
    codes = []
    if row["model_proba"] >= 0.6: codes.append("model_decline_risk")
    if row["visible"]: codes.append("visible_traffic")
    if row["stale"]: codes.append("stale_content")
    if row["ctr_gap"]: codes.append("ctr_below_expected")
    if row["low_engagement"]: codes.append("low_engagement")
    return ",".join(codes) if codes else "no_flag"

def action_for(row):
    if row["model_proba"] < 0.5 or not row["visible"]:
        return "monitor"
    if row["ctr_gap"] and row["low_engagement"]:
        return "refresh_and_review_ctr_and_engagement"
    if row["ctr_gap"]:
        return "refresh_and_review_ctr"
    if row["low_engagement"]:
        return "refresh_and_review_engagement"
    if row["stale"]:
        return "refresh"
    return "monitor"

test_p["reason_code"] = test_p.apply(reason_codes, axis=1)
test_p["action_label"] = test_p.apply(action_for, axis=1)
test_p["priority_score"] = test_p["model_proba"] * test_p["impressions_90d"]

ranked = test_p.sort_values("priority_score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

print("Action mix (ranked playbook, this test split):")
print(ranked["action_label"].value_counts())
print()
print("Top 5 by priority score:")
show_cols = ["rank", "content_id", "priority_score", "action_label", "reason_code",
             "impressions_90d", "model_proba"]
print(ranked[show_cols].head(5).to_string(index=False))

Action mix (ranked playbook, this test split):
action_label
monitor                   5522
refresh_and_review_ctr    1369
refresh                    224
Name: count, dtype: int64

Top 5 by priority score:
 rank           content_id  priority_score           action_label                                                         reason_code  impressions_90d  model_proba
    1 content_5fe46e04994d   343852.991700 refresh_and_review_ctr model_decline_risk,visible_traffic,stale_content,ctr_below_expected           517715     0.664174
    2 content_c84a0ab98e90   193936.704123 refresh_and_review_ctr               model_decline_risk,visible_traffic,ctr_below_expected           223271     0.868616
    3 content_73c54f78c06a   164727.590465 refresh_and_review_ctr               model_decline_risk,visible_traffic,ctr_below_expected           213963     0.769888
    4 content_8c19996aa890   158621.358918                monitor                                  visible_traffic,ctr_below_expected      

**How a reviewer uses this tomorrow.** Start at rank 1 and work down. For every
`refresh_and_review_ctr*` row, open the page and rule out a branded query, a mid-run title
experiment, or a SERP feature above the result before rewriting anything. For any `ctr == 0.00%`
row at real volume, check for a tracking gap first, a content problem second. `monitor` rows are
not "fine forever" — they're "not enough evidence yet," reviewed again next cycle.

**No-go list.** No auto-publishing, no automatic deprioritization, no cross-client comparison as
a performance judgment (the queue is traffic-weighted, not fairness-weighted), and no
client-facing use of these reason codes without human review.

## 7. Artifacts the paper embeds

In [7]:
import json

os.makedirs("work/outputs", exist_ok=True)

fig, ax = plt.subplots(figsize=(6.5, 4))
ranked["action_label"].value_counts().plot(kind="barh", ax=ax, color="#2b6cb0")
ax.set_title("Action mix — capstone playbook (test split)")
ax.set_xlabel("count")
fig.tight_layout()
fig.savefig("work/figures/capstone_action_mix.png", dpi=130)
plt.close(fig)

capstone_metrics = {
    "model": "logistic_regression_grouped_split",
    "split": "client_grouped (GroupShuffleSplit, test_size=0.25, seed=42)",
    "base_rate": round(float(test["is_declining_label"].mean()), 3),
    "results_vs_baseline": {k: {mk: (None if mv is None else round(float(mv), 3))
                                 for mk, mv in v.items()} for k, v in results.items()},
    "n_test_rows": int(len(test)),
    "n_clients_total": int(df["client_id"].nunique()),
    "action_mix": ranked["action_label"].value_counts().to_dict(),
}
with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(capstone_metrics, f, indent=2)

print("Saved work/figures/capstone_action_mix.png and work/outputs/capstone_metrics.json")
print(json.dumps(capstone_metrics, indent=2))

Saved work/figures/capstone_action_mix.png and work/outputs/capstone_metrics.json
{
  "model": "logistic_regression_grouped_split",
  "split": "client_grouped (GroupShuffleSplit, test_size=0.25, seed=42)",
  "base_rate": 0.517,
  "results_vs_baseline": {
    "baseline_rule": {
      "p@10": 0.3,
      "p@20": 0.3,
      "roc_auc": 0.489
    },
    "logistic_regression": {
      "p@10": 0.8,
      "p@20": 0.7,
      "roc_auc": 0.575
    },
    "random_forest": {
      "p@10": 0.5,
      "p@20": 0.5,
      "roc_auc": 0.597
    }
  },
  "n_test_rows": 7115,
  "n_clients_total": 32,
  "action_mix": {
    "monitor": 5522,
    "refresh_and_review_ctr": 1369,
    "refresh": 224
  }
}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and
  **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut
  + a 3-sentence employer-facing summary.

---
## ML-12 — Closing package

### 5-minute demo outline
1. **The question (30s).** Which of a client's content pages should get reviewed for a refresh
   first — and can I show my reasoning, not just a score?
2. **The data (30s).** 30,000 real content items across 32 clients, 90-day search performance
   window. Show the base rate: 54.2% of pages are already labeled declining — so "just guess
   declining" isn't a bar worth beating.
3. **The baseline (1 min).** A transparent rule: stale + visible + underperforming CTR for its
   own position tier. Show precision@10.
4. **The model (1.5 min).** Logistic regression on a client-grouped honest split. Show the
   results table: model beats baseline and base rate on precision@10/20.
5. **The catch (1 min).** Show the leakage test live: add the label-derived window columns back
   in, watch precision@10 jump to 1.00 — then explain why that's a red flag, not a win.
6. **The recommendation (30s).** Walk through rank 1 in the playbook: the reason codes, the
   action, and what a human must check before acting on it.

### Social post cut
Built a content-refresh priority model on 30,000 real FlyRank client pages. A simple rule
(stale + underperforming CTR) gets 30% precision on the top 10 flagged pages. Logistic
regression, validated honestly on clients it never saw, gets 80% — beating a 54% base rate. The
full paper, baseline, leakage checks, and ranked playbook are public: [link].

### Employer-facing summary
I built and honestly validated a content-refresh priority model on 30,000 real FlyRank client
pages (32 clients), comparing it against a transparent rule baseline on an identical,
client-grouped holdout split. The model lifts precision@10 from a 30% rule baseline to 80%, and
I verified that gap wasn't leakage by deliberately re-introducing the label-derived columns and
watching the score jump to a suspicious 100% — then removing them again.